# S4 · Ejercicio 5a — Ingeniería de características espacial (OXXO)

**Curso de Big Data · Dr. Abel Coronado** · Semana 4

> ⚙️ **Laboratorio BDP v7 (Codespaces o portable) · Spark 4.1 · Scala 2.13 · Sedona 1.9.1.**
> Un solo cuaderno para todas las plataformas, con el kernel **BDP v7 (Spark 4.1 · Python 3.12)**. Datos de práctica: recorte de Aguascalientes, cargado en HDFS con `python3 bdp_start.py datos`.

🎯 **Objetivo.** Construir el dataset OXXO vs Abarrotes con features espaciales (buffers + spatial joins de DENUE y Censo con Sedona).

📦 **Datos.** `/data/raw/geodatos_mexico/{censo_2020_nacional,denue_nacional}.parquet` (recorte de Aguascalientes).

✅ **Prerrequisitos.** Censo y DENUE en HDFS (`python3 bdp_start.py datos`). Con el recorte de un estado basta el driver de 2 GB.

📝 **Entregable.** `/data/processed/ml_dataset_final.geoparquet` en HDFS.

⬇️ **Repositorio:** https://github.com/abxda/bdp-lab-nube/tree/main/notebooks/semana_4

💬 **¿Un error?** Captura (celda + mensaje) y repórtalo por **Blackboard** (indica SO y paso).

---

> Lee los comentarios de cada celda: explican el porqué de cada paso.


In [ ]:
# =============================================================================
# Celda 1: Inicialización de Spark y Sedona
# =============================================================================
from pyspark.sql import SparkSession
import pyspark.sql.functions as F
from pyspark.sql.functions import col, expr, lower, substring, when, rand, lit, sum as _sum
from sedona.spark import SedonaContext

# Detener cualquier sesión previa para una inicialización limpia
try:
    spark.stop()
    print("Sesión de Spark anterior detenida.")
except:
    pass

print("» Paso 1: Configurando la sesión de Spark para un procesamiento geoespacial intensivo...")
spark = (
    SparkSession.builder.appName("FeatureEngineeringOXXO_HDFS")
    .config("spark.driver.memory", "2g")  # BDP v7: recorte de un estado
    .config("spark.locality.wait", "0")
    .config("spark.sql.shuffle.partitions", "200") # Optimización para los joins
    # BDP v7: Sedona 1.9.1 ya viene pre-cargado en el laboratorio (sin descargas de Maven)
    .config("spark.serializer", "org.apache.spark.serializer.KryoSerializer")
    .config("spark.kryo.registrator", "org.apache.sedona.core.serde.SedonaKryoRegistrator")
    .config("spark.sql.extensions", "org.apache.sedona.sql.SedonaSqlExtensions")
    .config("spark.sql.ansi.enabled", "false").getOrCreate()
)

sedona = SedonaContext.create(spark)
print("✅ Sesión de Spark con Apache Sedona iniciada correctamente.")

In [ ]:
# =============================================================================
# Celda 2: Carga de Datos y Preprocesamiento Inicial (Versión Corregida)
# =============================================================================
import pyspark.sql.functions as F
from pyspark.sql.functions import col, lower, substring, when

# --- LÍNEA DE CORRECCIÓN AÑADIDA AQUÍ ---
# Importamos los tipos de datos desde el módulo correcto
from pyspark.sql.types import IntegerType, DoubleType, LongType, FloatType

print("\n» Paso 2: Cargando y pre-procesando los datos base desde HDFS.")

# --- 2.1 Definir las rutas en HDFS ---
CENSO_PATH_HDFS = "hdfs:///data/raw/geodatos_mexico/censo_2020_nacional.parquet"
DENUE_PATH_HDFS = "hdfs:///data/raw/geodatos_mexico/denue_nacional.parquet"

# --- 2.2 Cargar DataFrames usando el formato 'geoparquet' ---
print(f"   Cargando Censo desde: {CENSO_PATH_HDFS}")
censo_sdf = spark.read.format("geoparquet").load(CENSO_PATH_HDFS)

print(f"   Cargando DENUE desde: {DENUE_PATH_HDFS}")
denue_raw_sdf = spark.read.format("geoparquet").load(DENUE_PATH_HDFS)


In [ ]:

# --- 2.3 Preprocesamiento del DENUE ---
print("   Procesando DENUE: creando 'est_per_ocu', 'codigo_act_2c', etc.")
denue_sdf = denue_raw_sdf.withColumn("codigo_act_2c", substring(col("codigo_act"), 1, 2)) \
    .withColumn("nom_estab", lower(col("nom_estab"))) \
    .withColumn("entidad", lower(col("entidad"))) \
    .withColumn("municipio", lower(col("municipio"))) \
    .withColumn("localidad", lower(col("localidad"))) \
    .withColumn("est_per_ocu",
        when(col("per_ocu").contains("0 a 5"), 2.5)
        .when(col("per_ocu").contains("6 a 10"), 8.0)
        .when(col("per_ocu").contains("11 a 30"), 20.5)
        .when(col("per_ocu").contains("31 a 50"), 40.5)
        .when(col("per_ocu").contains("51 a 100"), 75.5)
        .when(col("per_ocu").contains("101 a 250"), 175.5)
        .when(col("per_ocu").contains("251 y más"), 350.0)
        .otherwise(1.0)
    )

# --- 2.4 Preprocesamiento del Censo ---
# Reemplazar nulos por 0 en todas las columnas numéricas del censo
print("   Limpiando Censo: Reemplazando nulos por 0 en columnas numéricas.")
# AHORA ESTA LÍNEA FUNCIONARÁ CORRECTAMENTE
numeric_censo_cols = [f.name for f in censo_sdf.schema.fields if isinstance(f.dataType, (IntegerType, DoubleType, LongType, FloatType))]
censo_sdf = censo_sdf.na.fill(0, subset=numeric_censo_cols)

censo_count = censo_sdf.count()
denue_count = denue_sdf.count()

print(f"✅ Datos base cargados y pre-procesados. Censo: {censo_count:,} filas, DENUE: {denue_count:,} filas.")

In [ ]:
denue_sdf.show(n=2, truncate=False, vertical=True)

In [ ]:
# =============================================================================
# Celda 3: Creación del Dataset Balanceado (OXXO vs Abarrotes)
# =============================================================================
print("\n» Paso 3: Creando el dataset balanceado 'nacional_sdf'.")

# --- 3.1 Filtrar OXXOs y Abarrotes ---
oxxos_sdf = denue_sdf.filter(
    (col("codigo_act") == '462112') &
    (col("nom_estab").contains("oxxo")) &
    (~col("nom_estab").contains("distribuc"))
).withColumn("klass", lit(1))

abarrotes_sdf = denue_sdf.filter(col("codigo_act") == '461110').withColumn("klass", lit(0))

# --- 3.2 Muestreo Aleatorio y Unión ---
oxxo_count = oxxos_sdf.count()
print(f"   Número de OXXOs encontrados: {oxxo_count:,}")
abarrotes_sample_sdf = abarrotes_sdf.orderBy(rand(seed=42)).limit(oxxo_count)

# --- 3.3 Creación del DataFrame 'nacional' ---
nacional_cols = ["id", "klass", "codigo_act", "codigo_act_2c", "clee", "nom_estab", "cve_ent", "est_per_ocu", "geometry"]
# Renombramos 'est_per_ocu' a 'personal' para coincidir con tu script
nacional_sdf = oxxos_sdf.select(nacional_cols).withColumnRenamed("est_per_ocu", "personal") \
                      .unionByName(abarrotes_sample_sdf.select(nacional_cols).withColumnRenamed("est_per_ocu", "personal"))

print(f"✅ Dataset 'nacional_sdf' creado con {nacional_sdf.count():,} registros totales.")

In [ ]:
# =============================================================================
# Celda 4: Feature Engineering con Agregaciones Espaciales (Versión Corregida)
# =============================================================================
from pyspark.sql.types import IntegerType, DoubleType, LongType, FloatType

print("\n» Paso 4: Realizando Feature Engineering con Spatial Joins (esto puede tardar).")

# --- 4.1 Creación de Buffers---
print("   Creando buffers de 100m y 500m...")
nacional_con_buffers = nacional_sdf \
    .withColumn("buffer_100m", expr("ST_Buffer(geometry, 100)")) \
    .withColumn("buffer_500m", expr("ST_Buffer(geometry, 500)"))

# Esto evita que Spark tenga que recalcular los buffers cada vez.
print(f"   Buffers creados Total de filas: {nacional_con_buffers.count()}")

In [ ]:
# --- 4.2 Función para agregar features de DENUE ---
def aggregate_denue_features(base_df, denue_df, buffer_col_name, suffix):
    print(f"      Agregando features del DENUE para buffer {suffix}...")
    
    # Spatial Join
    joined = base_df.join(denue_df.alias("d"), expr(f"ST_Intersects({buffer_col_name}, d.geometry)"))
    
    # Agregación general de personal
    agg_epo = joined.groupBy(base_df.id).agg(_sum("d.est_per_ocu").alias(f"epo_{suffix}"))
    
    # Agregación por actividad y pivote
    act_codes = ['51', '54', '11', '22', '52', '71', '43', '31', '61', '46', '23', '55', '93', '53', '81', '33', '48', '32', '56', '49', '62', '21', '72']
    # CORRECCIÓN: Especificamos el DataFrame para el 'id' en groupBy
    agg_pivot = joined.groupBy(base_df.id).pivot("d.codigo_act_2c", act_codes).agg(_sum("d.est_per_ocu"))
    
    # Renombrar columnas pivotadas
    for code in act_codes:
        agg_pivot = agg_pivot.withColumnRenamed(code, f"act_{code}_{suffix}")
        
    # Unir ambas agregaciones
    return agg_epo.join(agg_pivot, "id", "outer")

# --- 4.3 Función para agregar features del Censo ---
def aggregate_censo_features(base_df, censo_df, buffer_col_name, suffix):
    print(f"      Agregando features del CENSO para buffer {suffix}...")
    
    # Spatial Join
    joined = base_df.join(censo_df.alias("c"), expr(f"ST_Intersects({buffer_col_name}, c.geometry)"))
    
    # Crear dinámicamente las expresiones de agregación
    numeric_censo_cols = [f.name for f in censo_df.schema.fields if isinstance(f.dataType, (IntegerType, DoubleType, LongType, FloatType))]
    agg_exprs = [_sum(f"c.{col}").alias(f"censo_{col}_{suffix}") for col in numeric_censo_cols]
    
    return joined.groupBy(base_df.id).agg(*agg_exprs)



In [ ]:
# --- 4.4 Ejecutar las agregaciones ---
denue_agg_100 = aggregate_denue_features(nacional_con_buffers, denue_sdf, "buffer_100m", "100")
denue_agg_500 = aggregate_denue_features(nacional_con_buffers, denue_sdf, "buffer_500m", "500")
censo_agg_100 = aggregate_censo_features(nacional_con_buffers, censo_sdf, "buffer_100m", "100")
censo_agg_500 = aggregate_censo_features(nacional_con_buffers, censo_sdf, "buffer_500m", "500")

print("✅ Agregaciones de entorno completadas.")

In [ ]:
# =============================================================================
# Celda 5: Unión Final, Limpieza y Guardado (Versión Corregida y Robusta)
# =============================================================================
print("\n» Paso 5: Uniendo todas las features en el dataset final.")

# --- 5.1 Unir todas las features de forma secuencial y limpia ---
# Empezamos con nuestro DataFrame base
final_sdf = nacional_sdf

# Unimos cada tabla de agregados una por una, usando una condición de join simple
# y especificando que el tipo de join es 'left_outer'
final_sdf = final_sdf.join(denue_agg_100, "id", "left_outer")
final_sdf = final_sdf.join(denue_agg_500, "id", "left_outer")
final_sdf = final_sdf.join(censo_agg_100, "id", "left_outer")
final_sdf = final_sdf.join(censo_agg_500, "id", "left_outer")

# --- 5.2 Limpieza Final ---
# Rellenar con 0 todos los valores nulos que resultaron de los left joins
final_sdf = final_sdf.na.fill(0)

# Restar el personal del propio establecimiento de las columnas agregadas.
# Es importante asegurarse de que las columnas pivotadas existan antes de intentar usarlas.
# Si un código de actividad no tenía ningún negocio en un buffer, la columna no se creará.
# Para hacerlo robusto, verificamos si la columna existe.
if "act_46_100" in final_sdf.columns:
    final_sdf = final_sdf.withColumn("act_46_100", col("act_46_100") - col("personal"))

if "act_46_500" in final_sdf.columns:
    final_sdf = final_sdf.withColumn("act_46_500", col("act_46_500") - col("personal"))

# Hacemos lo mismo para el total de personal ocupado (epo)
if "epo_100" in final_sdf.columns:
    final_sdf = final_sdf.withColumn("epo_100", col("epo_100") - col("personal"))

if "epo_500" in final_sdf.columns:
    final_sdf = final_sdf.withColumn("epo_500", col("epo_500") - col("personal"))


print("   Mostrando una muestra del dataset final para Machine Learning:")
# Preparamos una lista de columnas a mostrar, verificando que existan
cols_to_show = ["id", "klass", "personal"]
if "epo_100" in final_sdf.columns: cols_to_show.append("epo_100")
if "act_46_100" in final_sdf.columns: cols_to_show.append("act_46_100")
if "censo_POBTOT_100" in final_sdf.columns: cols_to_show.append("censo_POBTOT_100")

final_sdf.select(cols_to_show).show()


# --- 5.3 Guardar el resultado en HDFS ---
OUTPUT_PATH_HDFS = "hdfs:///data/processed/ml_dataset_final.geoparquet"
print(f"\nGuardando el dataset final enriquecido en: {OUTPUT_PATH_HDFS}")

final_sdf.write.format("geoparquet").mode("overwrite").save(OUTPUT_PATH_HDFS)

print("\n✅ ¡Proceso completado! El dataset para ML está listo en HDFS.")



In [ ]:
# =============================================================================
# Celda 6: Inspección del Dataset Final
# =============================================================================

print("\n» Paso 6: Inspeccionando el esquema del dataset final para Machine Learning.")

# --- 1. Definir la ruta del archivo de resultado en HDFS ---
FINAL_DATASET_PATH = "hdfs:///data/processed/ml_dataset_final.geoparquet"

# --- 2. Leer el dataset final desde HDFS ---
print(f"   Leyendo el resultado desde: {FINAL_DATASET_PATH}")
ml_dataset_sdf = spark.read.format("geoparquet").load(FINAL_DATASET_PATH)

# --- 3. Mostrar el Esquema Completo ---
print("\n--- Esquema Completo del DataFrame Final ---")
# .printSchema() es la forma estándar y más clara de ver la estructura
ml_dataset_sdf.printSchema()

# --- 4. Mostrar una Muestra de los Datos para Contexto ---
print("\n--- Muestra de los Datos Finales (primeras 5 filas) ---")
# Seleccionamos algunas columnas clave de cada categoría de features para verificar
# que la unión fue exitosa y los valores tienen sentido.
cols_to_show = [
    "id",
    "klass",
    "personal",
    "epo_100",          # Agregado DENUE general
    "act_46_100",       # Agregado DENUE por actividad
    "censo_POBTOT_100", # Agregado CENSO
    "epo_500",          # Agregado DENUE general (500m)
    "act_46_500",       # Agregado DENUE por actividad (500m)
    "censo_POBTOT_500"  # Agregado CENSO (500m)
]

# Filtramos las columnas que realmente existen en el DataFrame para evitar errores
existing_cols_to_show = [col_name for col_name in cols_to_show if col_name in ml_dataset_sdf.columns]

ml_dataset_sdf.select(existing_cols_to_show).show()


# --- 5. Opcional: Detener la sesión si ya no la necesitas ---
# print("\nSi has terminado tu análisis, puedes detener la sesión con spark.stop()")
# spark.stop()
# print("✅ Sesión de Spark detenida.")

In [ ]:
# --- 6. Detener la sesión de Spark ---
spark.stop()
print("✅ Sesión de Spark detenida.")